## Dash callbacks

### Notion de `decorateur`

Le décorateur est une fonction permettant de modifier le comportement d’autres fonctions, évitant la répétition de code.

In [1]:
def decorator(func):
    print('Python')
    return func

@decorator
def hello_world():
    print('hello world')

hello_world()

Python
hello world


In [2]:
def decorator(func):
    def wrapper():
        func()
        print('Appel de la fonction wrapper')
    return wrapper


@decorator
def func_2():
    print('Appel de la fonction 2')

func_2()

Appel de la fonction 2
Appel de la fonction wrapper


### Simple callback avec Dash

Les callbacks sont des fonctions qui sont appelées lors d'un changement de valeur dans l'interface utilisateur. Ils permettent ainsi la mise en relation entre les entrées et les sorties. La fonction `callback` s'utilise comme un décorateur et prend essentiellement les composant dash en paramètre tel que : '`Output`, `Input` et parfois `state`.

- `Output` : designe la ou l'une des sorties de la fontion décorée par le  callback. 

- `Input` : permet de récupérer les valeurs envoyées par l'utilisateur. Lorsqu'il est modifié, la fonction `callback` est appelée.

- `state` : permet de récupérer l'état courrant d'une composante.

In [4]:
import dash
from dash import dcc, html, Input, Output, callback

app = dash.Dash()

app.layout = html.Div([
    dcc.Input(id='input-1', value='dede', type='text'),
    html.Div(id='div-1', children='')
])

@callback(
    Output(component_id='div-1', component_property='children'),
    [Input(component_id='input-1', component_property='value')]
)
def update_div(input):
    return f'Introduit : {input}'

if __name__ == '__main__':
    app.run(debug=True, port=8050, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


### Exemple : Button callback

In [6]:
import dash
from dash import dcc, html, Input, Output, callback
from datetime import datetime

app = dash.Dash()

app.layout = html.Div([
    html.Button(id='btn-1', children='Button1', n_clicks_timestamp=0),
    html.Button(id='btn-2', children='Button2', n_clicks_timestamp=0),
    html.Div(id='div-1')
])

@callback(
    Output('div-1', 'children'),
    [Input('btn-1', 'n_clicks_timestamp'),
     Input('btn-2', 'n_clicks_timestamp')]
)
def displayClick(btn1, btn2):
    if int(btn1) > int(btn2) :
        msg = 'Button1 a été clické en dernier'
    elif int(btn2) > int(btn1) :
        msg = 'Button2 a été clické en dernier'
    else:
        msg = "Aucun bouton n'a été clické"
    return html.Div([
        html.Div(f'btn1: {datetime.fromtimestamp(btn1 / 1000)}'),
        html.Div(f'btn2: {datetime.fromtimestamp(btn2 / 1000)}'),
        html.Div(msg)
    ])

if __name__ == '__main__':
    app.run(debug=True, port=8050, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


### Exemple : dcc.Input and Button

In [ ]:
import dash
from dash import dcc, html, Input, Output, callback

app = dash.Dash()

app.layout = html.Div([

    html.Div([
        dcc.Input(id='input-1', type='text', value='Exemple de texte...')
    ]),

    html.Button(id='button-1', children='Soumettre', n_clicks=0),
    html.Div(id='div-1', children='Entrez votre texte et appuyez sur Soumettre')
])

@callback(
    Output(component_id='div-1', component_property='children'),
    [Input(component_id='input-1', component_property='value'),
     Input(component_id='button-1', component_property='n_clicks')]
)
def update_output(value, n_clicks):
    return f'Vous avez saisi : ``{value}`` et appuyé sur soumettre {n_clicks} fois'

if __name__ == '__main__':
    app.run(debug=True, port=8050, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


### Exemple : Dropdown Graph

In [ ]:
import dash
from dash import dcc, html, Input, Output, callback

app = dash.Dash()
app.layout = html.Div([
    dcc.Dropdown(
        id='drop-1',
        options=[
            {'label': 'Français', 'value': 'FR'},
            {'label': 'Américain', 'value': 'US'}
        ],
        value='FR'
    ),
    dcc.Graph( id='graph-1')
])

@callback(
    Output('graph-1', 'figure'),
    [Input('drop-1', 'value')]
)
def update_graph(value):
    data_dict = {
        'FR': [3, 2, 5, 4, 7],
        'US': [4, 1, 3, 4, 2]
    }
    return {'data': [
        {'y': data_dict[value],
         'type': 'scatter',
         'fill': 'tozeroy'}
    ]}

if __name__ == '__main__' :
    app.run(debug=True, port=8050, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
KeyError: None

---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
KeyError: None



### Exemple : Graph callback

In [ ]:

import dash
from dash import dcc, html, Input, Output, callback
import plotly.graph_objects as go 
import pandas as pd

# df = pd.read_csv('https://ml-repository-krakers.s3-eu-west-1.amazonaws.com/dash_course/data.csv')
df = pd.read_csv('country_data.csv')

app = dash.Dash()

app.layout = html.Div([
    dcc.Graph(id='graph-1'),
    dcc.Slider(
        id='slider-1',
        min=df.year.min(),
        max=df.year.max(),
        value=df.year.min(),
        marks={str(year): str(year) for year in df.year.unique()},
        step=None
    )
])

@callback(
    Output('graph-1', 'figure'),
    [Input('slider-1', 'value')]
)
def update_graph(seleceted_year):
    dff = df.query(f'year == {seleceted_year}')
    traces = []
    for cont in df.continent.unique():
        dff_cont = dff[dff.continent == cont]
        traces.append(
            go.Scatter(
                x=dff_cont.gdpPercap,
                y=dff_cont.lifeExp,
                mode='markers',
                name=cont,
                opacity=0.6,
                marker={
                    'size': 15,
                    'line': {'width': 1.5, 'color': 'white'}
                }
            )
        )

    return {
        'data': traces,
        'layout': go.Layout(
            title_text='Graphique interactif',
            xaxis={'type': 'log', 'title': 'PIB par tête'},
            yaxis={'title': 'Espérance de vie'},
            hovermode='closest'
        )
    }

if __name__ == '__main__':
    app.run(debug=True, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


### Exemple : Cascades select avec le composant dcc.RadioItems

In [ ]:
import dash
from dash import dcc, html, Input, Output, callback

app = dash.Dash()

app.layout = html.Div([
    dcc.RadioItems(
        id='radio-1',
        options=[
            {'label': 'France', 'value': 'France'},
            {'label': 'États-Unis', 'value': 'États-Unis'}
        ],
        value='France'
    ),
    html.Hr(),
    dcc.RadioItems( id='radio-2' ),
    html.Hr(),
    html.Div(id='div-1')
])

@callback(
    Output(component_id='radio-2', component_property='options'),
    [Input(component_id='radio-1', component_property='value')]
)
def set_radio_2_options(selected_country):
    if selected_country == 'France' :
        return [
            {'label': 'Nantes', 'value': 'Nantes'},
            {'label': 'Paris', 'value': 'Paris'},
            {'label': 'Marseille', 'value': 'Marseille'}
        ]
    elif selected_country == 'États-Unis':
        return [
            {'label': 'New York', 'value': 'New York'},
            {'label': 'Los Angeles', 'value': 'Los Angeles'}
        ]
    else:
        raise ValueError('Valeur invalide !')

@callback(
    Output(component_id='radio-2', component_property='value'),
    [Input(component_id='radio-1', component_property='value')]
)
def set_city(selected_country):
    if selected_country == 'France':
        return 'Nantes'
    elif selected_country == 'États-Unis':
        return 'New York'
    else:
        raise ValueError('Valeur invalide !')

@callback(
    Output(component_id='div-1', component_property='children'),
    [Input(component_id='radio-1', component_property='value'),
     Input(component_id='radio-2', component_property='value')]
)
def set_div(input1, input2):
    if input2 is None:
        return ''
    return f'Vous avez entré {input1} et {input2}'

if __name__ == '__main__' :
    app.run(debug=True, port=8050, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


### PreventUpdate

Permet de bloquer le rafraichissement du composant

In [ ]:
import dash
from dash import dcc, html, Input, Output, callback
import plotly.graph_objects as go 
from dash.exceptions import PreventUpdate

app = dash.Dash()

app.layout = html.Div([
    html.Button('Clicker ici', id='button-1'),
    html.Div(id='div-1')
])

@callback(
    Output('div-1', 'children'),
    [Input('button-1', 'n_clicks')]
)
def update_output(n_clicks):
    return f'Des informations très précieuses ! Vous avez cliqué {n_clicks} fois!'
    # if n_clicks is None:
    #     raise PreventUpdate
    # else:
    #     return f'Des informations très précieuses ! Vous avez cliqué {n_clicks} fois!'
    

if __name__ == '__main__' :
    app.run(debug=True, port=8050, jupyter_mode='external')

### Exemple : Tabs

In [9]:
import dash
from dash import dcc, html, Input, Output, callback

app = dash.Dash()

app.layout = html.Div([
    html.H2('Dash Tab Template'),
    dcc.Tabs(
        id='tabs-1',
        children=[
            dcc.Tab(label='Bar Plot', value='tab-1'),
            dcc.Tab(label='Line Plot', value='tab-2'),
            dcc.Tab(label='Scatter Plot', value='tab-3')
        ],
        value='tab-1'
    ),

    html.Div(id='div-1')
])

@callback(
    Output('div-1', 'children'),
    [Input('tabs-1', 'value')]
)
def render_content(tab):
    if tab == 'tab-1' :
        return html.Div([
            html.H3('Bar Plot Content'),
            dcc.Graph(
                figure={
                    'data': [
                        {'x': [1, 2, 3],
                         'y': [2, 1, 3],
                         'type': 'bar'}
                    ]
                }
            )
        ])
    elif tab == 'tab-2' :
        return html.Div([
            html.H3('Line Plot Content'),
            dcc.Graph(
                figure={
                    'data': [
                        {'x': [1, 2, 3, 4, 5],
                         'y': [2, 1, 3, 4, 2],
                         'type': 'line'}
                    ]
                }
            )
        ])
    elif tab == 'tab-3' :
        return html.Div([
            html.H3('Scatter Plot Content'),
            dcc.Graph(
                figure={
                    'data': [
                        {'x': [1, 2, 3, 4, 5],
                         'y': [2, 1, 3, 3, 2],
                         'mode': 'markers'}
                    ]
                }
            )
        ])


if __name__ == '__main__':
    app.run(debug=True, port=8050, jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/


### Exemple : upload file

In [ ]:

import dash
from dash import dcc, html, Input, Output, callback, State
import plotly.graph_objects as go 
import base64
import io
import pandas as pd
from datetime import datetime
import dash_table


app = dash.Dash()

app.layout = html.Div([
    dcc.Upload(
        id='upload-1',
        children=html.Div([
            'Drag and drop or ',
            html.A('Select Files')
        ]),
        style={
            'width': '100%',
            'height': '60px',
            'textAlign': 'center',
            'borderWidth': '1px',
            'borderStyle': 'dashed',
            'borderRadius': '5px',
            'lineHeight': '60px'
        },
        multiple=True
    ),

    html.Div(id='div-1')
])

def parse_contents(content, name, date):
    content_type, content_string = content.split(',')

    decoded = base64.b64decode(content_string)
    try:
        if name.endswith('.csv') :
            df = pd.read_csv(io.StringIO(decoded.decode('utf-8')))
        elif name.endswith('.xls') or name.endswith('.xlsx') :
            df = pd.read_excel(io.BytesIO(decoded))
    except Exception as e:
        print(e)
        return html.Div([
            "Une erreur s'est produite lors du traitement du fichier."
        ])

    return html.Div([
        html.H5(name),
        html.H6(datetime.fromtimestamp(date)),

        dash_table.DataTable(
            data=df.to_dict('records'),
            columns=[{'name': col, 'id': col} for col in df.columns]
        ),
        html.Hr(),
        html.Div('Raw Content'),
        html.Pre(content[:300] + '...')
    ])


@callback(
    Output('div-1', 'children'),
    [Input('upload-1', 'contents')],
    [State('upload-1', 'filename'),
     State('upload-1', 'last_modified')]
)
def update_output(list_of_contents, list_of_names, list_of_dates):
    print(list_of_contents)
    print(list_of_names)
    print(list_of_dates)
    if list_of_contents is not None:
        children = [
            parse_contents(content, name, date) for content, name, date in zip(list_of_contents,
                                                                               list_of_names,
                                                                               list_of_dates)
        ]
        return children


if __name__ == '__main__':
    app.run(debug=True, port=8050, jupyter_mode='external')